# 01.03 Google Gemini · Vertex AI (통합 SDK 4.0)

`langchain-google-genai 4.x`는 **Gemini API(소비자/개발자 키)**와 **Vertex AI(엔터프라이즈 GCP)** 두 경로를 단일 클래스 `ChatGoogleGenerativeAI` 하나로 제공한다. `vertexai=True` 플래그로 라우팅을 전환한다.

## 학습 목표

- `ChatGoogleGenerativeAI(model="gemini-2.5-flash")` 기본 사용
- `vertexai=True, project=..., location="us-central1"`로 Vertex 경로 전환
- `bind_tools` / `with_structured_output` 동작 확인
- Gemini 전용 **safety_settings**, **response_mime_type** 옵션 이해

## 언제 쓰나

- Google AI Studio 키(`GOOGLE_API_KEY`)만 있는 빠른 프로토타입
- GCP 프로젝트 + IAM 기반 엔터프라이즈(Vertex AI)
- 멀티모달(이미지/비디오) 입력이 많은 워크로드

## 01.03.1 환경 설정

In [ ]:
# !pip install -U langchain langchain-google-genai
import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("GOOGLE_API_KEY"), "GOOGLE_API_KEY 필요 (Vertex 모드면 GOOGLE_APPLICATION_CREDENTIALS 도 가능)"

## 01.03.2 기본 사용 — Gemini API 경로

`GOOGLE_API_KEY` 하나면 끝. 모델 ID는 `gemini-2.5-flash`, `gemini-2.5-pro` 등.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
msg = model.invoke("Gemini와 Vertex AI가 하나의 SDK로 통합된 이유를 한 문단으로 설명해줘.")
print(msg.content)
print("usage:", msg.usage_metadata)

## 01.03.3 `init_chat_model()` 경로

In [ ]:
from langchain.chat_models import init_chat_model

m = init_chat_model("google_genai:gemini-2.5-flash")
print(m.invoke("핑").content[:80])

## 01.03.4 Vertex AI 경로 — 엔터프라이즈 라우팅

`vertexai=True`를 주면 `project` + `location` 을 읽어 Vertex AI 백엔드로 라우팅한다. 자격은 `gcloud auth application-default login` 또는 `GOOGLE_APPLICATION_CREDENTIALS` 서비스 계정 JSON. 이 셀은 자격이 없으면 정보 출력만 한다.

In [ ]:
have_vertex = bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS") or os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("Vertex 자격 감지:", have_vertex)

if have_vertex:
    vertex = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        vertexai=True,
        project=os.environ.get("GOOGLE_CLOUD_PROJECT"),
        location="us-central1",
    )
    print("vertex →", vertex.invoke("ping").content[:60])
else:
    print("Vertex 경로는 자격 없음 — Gemini API 경로로 충분히 학습 가능")

## 01.03.5 Tool calling

Gemini는 JSON schema 기반 function calling 을 기본 지원한다.

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""
    return a + b

bound = model.bind_tools([add])
out = bound.invoke("13과 29를 더해줘")
print("tool_calls:", out.tool_calls)

## 01.03.6 Structured output + `response_mime_type`

In [ ]:
from pydantic import BaseModel

class Poem(BaseModel):
    title: str
    body: str

structured = model.with_structured_output(Poem)
print(structured.invoke("가을에 대한 짧은 시를 JSON으로 작성해줘."))

# JSON 전용 모드를 직접 지정할 수도 있다 (response_mime_type)
json_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    model_kwargs={"response_mime_type": "application/json"},
)
print("raw json →", json_model.invoke("{\"city\": \"Seoul\"} 형태로 응답해줘").content[:120])

## 01.03.7 Safety settings

Gemini는 응답/프롬프트 양쪽에 안전 필터가 기본 적용된다. 민감 카테고리마다 임계값(`BLOCK_NONE`, `BLOCK_ONLY_HIGH`, ...)을 설정할 수 있다.

In [ ]:
from langchain_google_genai import HarmBlockThreshold, HarmCategory

safe = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    safety_settings={
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
    },
)
print(safe.invoke("짧게 인사해줘").content[:80])

## 체크리스트

- [ ] Gemini API(키) vs Vertex AI(`vertexai=True + project + location`) 라우팅을 구분했다
- [ ] `bind_tools`와 `with_structured_output`의 동작을 확인했다
- [ ] `safety_settings`, `response_mime_type` 으로 Gemini 고유 제어점을 찾았다

## 다음

- `01_openai.ipynb` — OpenAI / Azure
- `05_bedrock.ipynb` — 다른 엔터프라이즈(AWS) 배포 경로

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai